# 07 - Results Visualization

Portfolio-level dashboards and export-ready figures using RiskCharts.

## Objectives
- Use RiskCharts for all chart types
- Build portfolio-level dashboards
- Create export-ready figures

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import json
import joblib
import numpy as np
import pandas as pd

from src.data import FeatureEngineer
from src.mcda import ProjectRanker
from src.visualization.risk_charts import RiskCharts

PROCESSED_DIR = Path('../data/processed')
MODELS_DIR    = Path('../models/ml')

# All HTML exports go here so they don't clutter the notebooks directory
REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(exist_ok=True)

print('Imports OK')

Imports OK


## 1. Run Full Pipeline

We must run the full pipeline here (rather than just loading the CSV) because:
- `mcda_score` and `rank` come from `ProjectRanker` and are not persisted to disk
- `ml_risk_score` (continuous 0–1 probability) comes from the trained model
- `FeatureEngineer` produces the 12 derived features needed by MCDA criteria (SPI, blocker_ratio, defect_rate)

This mirrors what notebooks 05 and 06 do, ensuring all charts reflect the same data.

In [2]:
df = pd.read_csv(PROCESSED_DIR / 'jira_projects.csv')

# FeatureEngineer creates SPI, blocker_ratio, defect_rate and 9 other derived
# features that MCDA criteria depend on. They are not stored in the CSV.
fe = FeatureEngineer()
df = fe.create_features(df)

# Load the model trained by notebook 03 and score all 640 projects.
# We use the FULL dataset (not a train/test split) because this notebook's
# purpose is portfolio visualisation, not predictive evaluation.
model_path = MODELS_DIR / 'best_model.pkl'
feature_names_path = MODELS_DIR / 'feature_names.json'

if model_path.exists() and feature_names_path.exists():
    model = joblib.load(model_path)
    with open(feature_names_path) as f:
        feature_names = json.load(f)

    X = df[feature_names].copy()
    # Cap reopen_rate to match the training-time preprocessing in notebook 03
    if 'reopen_rate' in X.columns:
        X['reopen_rate'] = X['reopen_rate'].clip(upper=1.0)
    X = X.fillna(0).replace([np.inf, -np.inf], 0)

    # predict_proba[:, 1] = P(High Risk) — used as the primary MCDA cost criterion
    df['ml_risk_score'] = model.predict_proba(X)[:, 1]
    print(f'ML scores generated using: {type(model).__name__}')
else:
    # Fallback if notebook 03 hasn't been run yet
    print('No saved model — using risk_score_composite as ml_risk_score proxy')
    df['ml_risk_score'] = df['risk_score_composite'].clip(0, 1)

# Real LLM sentiment scores from notebook 04 (gpt-4.1, run 2026-03-22).
# 49 of 50 batch projects have scores; ActiveMQ Website excluded (no status_comments).
# The remaining 591 projects default to 0.0 (neutral) until a full run is done.
# Storing them here avoids re-calling the API on every notebook run (cost saving).
LLM_SENTIMENT = {
    # 50-project batch from notebook 04 (gpt-4.1, run 2026-03-22).
    # ActiveMQ Website excluded — no status_comments text available.
    'Abdera':                            0.15,
    'Accumulo':                          0.10,
    'ACE':                               0.15,
    'ActiveCluster':                     0.10,
    'Addressing':                       -0.60,
    'Apache AGE':                       -0.35,
    'Apache AGE (old)':                  0.10,
    'Agila':                             0.20,
    'Airavata':                          0.20,
    'ALOIS':                             0.40,
    'AMATERASU':                         0.30,
    'Ambari':                           -0.40,
    'ActiveMQ Classic':                 -0.25,
    'ActiveMQ CLI Tools':                0.20,
    'ActiveMQ C++ Client':              -0.60,
    'ActiveMQ .Net':                    -0.20,
    'Anakia':                            0.10,
    'Apache Any23 (Retired)':            0.10,
    'Portals Apps (Retired)':            0.20,
    'Apache Apex Core':                  0.10,
    'Apache Apex Malhar':               -0.20,
    'APISIX':                            0.40,
    'ActiveMQ Apollo (Retired)':        -0.60,
    'WSRF':                              0.10,
    'Maven Archetype (Moved to GitHub Issues)': -0.60,
    'AriaTosca':                         0.10,
    'Aries':                            -0.25,
    'AltRMI':                           -0.10,
    'Apache Arrow':                      0.15,
    'ActiveMQ Artemis':                 -0.35,
    'www.apache.org website':            0.20,
    'Apache AsterixDB':                 -0.40,
    'Asyncweb':                         -0.35,
    'Atlas':                             0.15,
    'Attic':                            -0.30,
    'Commons Attributes':               -0.30,
    'Aurora':                           -0.25,
    'Tiles Autotag':                    -0.60,
    'Avalon':                           -0.70,
    'Apache Avro':                       0.25,
    'Apache AWF':                        0.25,
    'Axiom':                             0.10,
    'Axis':                             -0.35,
    'Axis2':                            -0.15,
    'Axis2-C':                           0.20,
    'Axis-C++':                         -0.20,
    'Bahir (Retired)':                   0.10,
    'BatchEE':                           0.15,
    'Batik':                            -0.30,
}
df['sentiment_score'] = df['project_name'].map(LLM_SENTIMENT).fillna(0.0)
n_llm = (df['sentiment_score'] != 0.0).sum()
print(f'Sentiment scores: {n_llm} from LLM (50-project batch), {len(df) - n_llm} defaulted to 0.0 (neutral)')

# Run MCDA to get the continuous mcda_score and ordinal rank for all projects.
# The 10 non-zero sentiment values will now influence the 25% LLM weight in TOPSIS.
ranker  = ProjectRanker()
rankings = ranker.rank(df)

# Merge MCDA output back so every chart can use both ML and MCDA columns
df = df.merge(
    rankings[['project_id', 'mcda_score', 'rank', 'risk_level']].rename(
        columns={'risk_level': 'mcda_risk_level'}
    ),
    on='project_id'
)

print(f'\nPipeline complete: {len(df)} projects, {len(df.columns)} columns')
print(f'MCDA score range: [{df["mcda_score"].min():.3f}, {df["mcda_score"].max():.3f}]')
print(f'Risk distribution (MCDA):\n{df["mcda_risk_level"].value_counts()}')

2026-03-22 13:26:28.818 | INFO     | src.data.feature_engineer:create_features:80 - Created 12 derived features
2026-03-22 13:26:29.006 | INFO     | src.mcda.topsis:fit:120 - TOPSIS fitted on 640 alternatives with 5 criteria
2026-03-22 13:26:29.010 | INFO     | src.mcda.ranker:rank:112 - Ranked 640 projects


ML scores generated using: LGBMClassifier
Sentiment scores: 49 from LLM (50-project batch), 591 defaulted to 0.0 (neutral)

Pipeline complete: 640 projects, 65 columns
MCDA score range: [0.380, 0.963]
Risk distribution (MCDA):
mcda_risk_level
Medium    218
High      211
Low       211
Name: count, dtype: int64


## 2. Risk Distribution Pie Chart

In [3]:
# Use MCDA risk_level (percentile-based: bottom third = High, top third = Low)
# rather than the binary ML label, because MCDA gives three balanced tiers
# across all 640 projects.
#
# We pass only a single-column DataFrame rather than renaming mcda_risk_level on
# the full df — the full df already has a 'risk_level' column (the ML label from
# the CSV), so renaming would produce two identically-named columns and crash pandas.
pie_df = df[['mcda_risk_level']].rename(columns={'mcda_risk_level': 'risk_level'})
fig = RiskCharts.risk_distribution_pie(pie_df)
fig.show()

out = REPORTS_DIR / 'risk_distribution.html'
fig.write_html(str(out))
print(f'Saved → {out}')

Saved → ../reports/risk_distribution.html


## 3. Risk Score Bar Chart

In [4]:
# mcda_score is always available after the pipeline cell above.
# RiskCharts.risk_score_bar uses nsmallest(top_n, score_col) internally —
# correct for MCDA scores where lower = riskier (closer to anti-ideal solution).
fig = RiskCharts.risk_score_bar(df, top_n=15, score_col='mcda_score')
fig.show()

out = REPORTS_DIR / 'risk_score_bar.html'
fig.write_html(str(out))
print(f'Saved → {out}')

Saved → ../reports/risk_score_bar.html


## 3b. Feature Importance

In [5]:
if not model_path.exists():
    print('No trained model found — run notebook 03 first.')
else:
    # feature_importances_ from LGBMClassifier are raw split counts (not normalised).
    # Sorting descending before passing to the chart ensures the plot shows the
    # genuinely most influential features rather than the first N alphabetically.
    imp_df = pd.DataFrame({'feature': feature_names, 'importance': model.feature_importances_})
    imp_df = imp_df.sort_values('importance', ascending=False)

    fig = RiskCharts.feature_importance_bar(imp_df, top_n=15)
    fig.show()

    out = REPORTS_DIR / 'feature_importance.html'
    fig.write_html(str(out))
    print(f'Saved → {out}')

Saved → ../reports/feature_importance.html


## 4. Risk Gauge and Radar Charts

In [6]:
# --- Portfolio risk gauge ---
# The mean MCDA score represents the portfolio's average "closeness to the ideal
# solution". risk_gauge() treats its input as a safety score (higher = safer) and
# inverts it internally: gauge_value = (1 - mean_mcda_score) × 100.
#
# NOTE: TOPSIS produces scores skewed toward higher values (most projects are not
# extreme outliers), so the gauge will typically show a lower % than the fraction
# of High-risk projects might suggest.  Print the actual High-risk % alongside
# the gauge so they can be interpreted together.
portfolio_safety    = float(df['mcda_score'].mean())
pct_high = (df['mcda_risk_level'] == 'High').mean()
fig_gauge = RiskCharts.risk_gauge(portfolio_safety, title='Portfolio Risk Score (MCDA)')
fig_gauge.show()
print(f'Portfolio mean MCDA score  : {portfolio_safety:.3f}')
print(f'Gauge displays             : {(1-portfolio_safety)*100:.1f}% risk')
print(f'Actual High-risk projects  : {pct_high:.1%} of portfolio ({int(pct_high*len(df))} projects)')
print(f'(The gauge reflects the MCDA score distribution, not the raw High-risk %;')
print(f' use the pie chart in section 2 for the clearer risk-tier breakdown.)')

# --- Portfolio risk profile radar ---
# Each axis represents one JIRA-derived risk dimension as a percentage (0–100%).
# All five use the same 0-100 scale so the radar shape is meaningful:
#
#   Defect Rate %    = bug_count / total_issues × 100
#   Blocker Ratio %  = blocker_count / total_issues × 100
#   Reopen Rate %    = reopen_count / total_issues × 100
#   Churn Rate %     = status_changes / total_issues × 100  (can exceed 100 on paper
#                      but is clipped; ~94% means ≈0.94 status changes per issue on avg)
#   Completion Gap % = 100 − completion_rate  (completion_rate stored as 0–100 in CSV)
#
# A large spike on Churn Rate is typical for active projects with many workflow transitions.
risk_dims = {
    'Defect Rate %':    round(df['defect_rate'].mean()   * 100, 1),
    'Blocker Ratio %':  round(df['blocker_ratio'].mean() * 100, 1),
    'Reopen Rate %':    round(df['reopen_rate'].mean()   * 100, 1),
    'Churn Rate %':     round(df['churn_rate'].mean()    * 100, 1),
    'Completion Gap %': round(100 - df['completion_rate'].mean(), 1),
}
print(f'\nPortfolio risk profile (all values in %):')
for k, v in risk_dims.items():
    print(f'  {k:<22}: {v:.1f}%')

fig_radar = RiskCharts.risk_category_radar({k: int(v) for k, v in risk_dims.items()})
fig_radar.show()

out = REPORTS_DIR / 'portfolio_risk_radar.html'
fig_radar.write_html(str(out))
print(f'\nSaved → {out}')

Portfolio mean MCDA score  : 0.796
Gauge displays             : 20.4% risk
Actual High-risk projects  : 33.0% of portfolio (211 projects)
(The gauge reflects the MCDA score distribution, not the raw High-risk %;
 use the pie chart in section 2 for the clearer risk-tier breakdown.)

Portfolio risk profile (all values in %):
  Defect Rate %         : 45.3%
  Blocker Ratio %       : 2.4%
  Reopen Rate %         : 7.5%
  Churn Rate %          : 94.3%
  Completion Gap %      : 24.4%



Saved → ../reports/portfolio_risk_radar.html


## 5. Project Comparison Radar

Compare the top-3 riskiest projects against the top-3 safest on normalised metrics.
This highlights *why* the risky projects rank low — which dimensions are worst.

In [7]:
# Metrics to compare — all risk-oriented (higher = worse) except completion_rate.
# We normalise each to [0, 1] using min-max so the radar axes are comparable.
compare_metrics = ['defect_rate', 'blocker_ratio', 'reopen_rate', 'churn_rate', 'ml_risk_score']

# Select the 3 most and 3 least risky projects by MCDA rank
top3_risky = df.nsmallest(3, 'mcda_score')
top3_safe  = df.nlargest(3, 'mcda_score')
sample     = pd.concat([top3_risky, top3_safe])

# Min-max normalisation uses the full portfolio range so the individual project
# values are relative to the entire dataset, not just these 6 projects.
normalised = []
for _, row in sample.iterrows():
    p = {'project_name': row['project_name']}
    for m in compare_metrics:
        col_min = df[m].min()
        col_max = df[m].max()
        # Avoid division by zero for zero-variance columns
        p[m] = float((row[m] - col_min) / (col_max - col_min)) if col_max > col_min else 0.0
    normalised.append(p)

# comparison_radar supports up to 4 projects; we show 3+3=6 so split into two charts
fig_risky = RiskCharts.comparison_radar(normalised[:3],  compare_metrics)
fig_safe  = RiskCharts.comparison_radar(normalised[3:],  compare_metrics)
fig_risky.update_layout(title='Top 3 Highest-Risk Projects — Metric Profile')
fig_safe.update_layout(title='Top 3 Lowest-Risk Projects — Metric Profile')

fig_risky.show()
fig_safe.show()

out_r = REPORTS_DIR / 'comparison_radar_risky.html'
out_s = REPORTS_DIR / 'comparison_radar_safe.html'
fig_risky.write_html(str(out_r))
fig_safe.write_html(str(out_s))
print(f'Saved → {out_r}')
print(f'Saved → {out_s}')

Saved → ../reports/comparison_radar_risky.html
Saved → ../reports/comparison_radar_safe.html


## 6. Sentiment Score Distribution

Shows how LLM sentiment is distributed across the projects that were analysed in
notebook 04. When all 640 projects have been scored, this distribution helps
validate whether the LLM is producing meaningful signal (spread between −1 and +1)
vs collapsing to neutral (all 0.0 from the default).

In [8]:
llm_analysed = df[df['sentiment_score'] != 0.0].copy()
n_analysed = len(llm_analysed)

if n_analysed == 0:
    print('No LLM sentiment scores available (all 0.0 default).')
    print('Run notebook 04 on all 640 projects to populate real sentiment values.')
else:
    print(f'Showing sentiment distribution for {n_analysed} LLM-analysed projects')
    print(f'(Remaining {len(df) - n_analysed} projects defaulted to 0.0 and excluded from histogram)\n')

    # We plot only the LLM-analysed projects so the histogram reflects real signal,
    # not the 630 neutral defaults which would collapse every other bin to zero.
    fig_sent = RiskCharts.sentiment_distribution(llm_analysed)
    fig_sent.update_layout(
        title=f'LLM Sentiment Distribution ({n_analysed} projects analysed with gpt-4.1)'
    )
    fig_sent.show()

    out = REPORTS_DIR / 'sentiment_distribution.html'
    fig_sent.write_html(str(out))
    print(f'Saved → {out}')
    print(f'\nSentiment range : [{llm_analysed["sentiment_score"].min():.2f}, {llm_analysed["sentiment_score"].max():.2f}]')
    print(f'Mean sentiment  : {llm_analysed["sentiment_score"].mean():.2f}  '
          f'(negative = risk-flagging language; positive = progress language)')
    print(f'\nPer-project scores:')
    for _, row in llm_analysed.sort_values('sentiment_score').iterrows():
        label = '▼ negative' if row['sentiment_score'] < 0 else '▲ positive'
        print(f'  {row["project_name"]:<30} {row["sentiment_score"]:+.2f}  {label}')

Showing sentiment distribution for 49 LLM-analysed projects
(Remaining 591 projects defaulted to 0.0 and excluded from histogram)



Saved → ../reports/sentiment_distribution.html

Sentiment range : [-0.70, 0.40]
Mean sentiment  : -0.08  (negative = risk-flagging language; positive = progress language)

Per-project scores:
  Avalon                         -0.70  ▼ negative
  Maven Archetype (Moved to GitHub Issues) -0.60  ▼ negative
  Tiles Autotag                  -0.60  ▼ negative
  Addressing                     -0.60  ▼ negative
  ActiveMQ Apollo (Retired)      -0.60  ▼ negative
  ActiveMQ C++ Client            -0.60  ▼ negative
  Apache AsterixDB               -0.40  ▼ negative
  Ambari                         -0.40  ▼ negative
  Axis                           -0.35  ▼ negative
  Apache AGE                     -0.35  ▼ negative
  Asyncweb                       -0.35  ▼ negative
  ActiveMQ Artemis               -0.35  ▼ negative
  Commons Attributes             -0.30  ▼ negative
  Attic                          -0.30  ▼ negative
  Batik                          -0.30  ▼ negative
  Aries                          

## Summary

### Purpose
Notebook 07 produces export-ready Plotly HTML visualisations for the full portfolio.
All charts are saved to `reports/` and can be embedded in presentations or a Streamlit dashboard.

### Charts produced
| File | Content | Source column |
|---|---|---|
| `risk_distribution.html` | Donut chart of High / Medium / Low project counts | `mcda_risk_level` |
| `risk_score_bar.html` | Top-15 riskiest projects horizontal bar | `mcda_score` |
| `feature_importance.html` | Top-15 ML features by split-count importance | `model.feature_importances_` (sorted) |
| `portfolio_risk_radar.html` | Five JIRA risk dimensions as portfolio averages (0–100%) | `defect_rate`, `blocker_ratio`, `reopen_rate`, `churn_rate`, `completion_rate` |
| `comparison_radar_risky.html` | Top-3 riskiest projects — normalised metric profile | MCDA rank |
| `comparison_radar_safe.html` | Top-3 safest projects — normalised metric profile | MCDA rank |
| `sentiment_distribution.html` | LLM sentiment histogram for the 49 gpt-4.1-analysed projects (50-project batch) | `sentiment_score` |

### Design decisions
| Decision | Choice | Reason |
|---|---|---|
| Run full pipeline | FeatureEngineer + ML + MCDA in Cell 3 | `mcda_score` is not persisted to disk; without it the bar chart silently fails |
| MCDA risk_level for pie chart | Percentile-based tiers via `mcda_risk_level` | Produces a balanced 33/33/33 split; avoids the duplicate `risk_level` column from the CSV |
| Consistent radar scaling | All axes 0–100% | Original code used different multipliers per axis (×10 for some, ×100 for others) |
| Sort feature importance | `sort_values` before `head()` | Without sorting, chart returned first-N insertion-order features, not top N by importance |
| Hard-code 49 LLM scores | `LLM_SENTIMENT` dict | Avoids re-calling the API on every notebook run; notebook 04 50-project batch was run on 2026-03-22 at ~$0.05 total |
| Sentiment histogram — LLM projects only | Filter `sentiment_score != 0.0` | The 591 neutral defaults would dominate the histogram and hide the real signal from the 49 analysed projects |

### Key findings from this run
- **MCDA risk split**: 211 High / 218 Medium / 211 Low — balanced, consistent with notebooks 05/06
- **Portfolio gauge**: Mean MCDA score 0.796 → 20.4% gauge risk. TOPSIS scores are high for most projects so this understates the 33% High-risk fraction; see pie chart for a clearer picture
- **Top risk dimensions**: Churn Rate (94.3%) dominates the radar — ~0.94 status changes per issue on average. Defect Rate (45.3%) is second. Blocker Ratio (2.4%) is low in absolute terms but was identified as the highest-sensitivity MCDA criterion in notebook 05
- **LLM sentiment (49 projects, 50-project batch)**: Range [−0.70, +0.40], mean −0.077. Avalon most negative (−0.70, high risk — project shutting down), ALOIS and APISIX most positive (+0.40, low/medium risk). 17/50 LLM-flagged as high risk vs 21/50 ML-flagged as high risk. Agreement rate 42% (21/50) — ML and LLM use complementary signals, justifying the hybrid MCDA approach

### Next steps
1. Run notebook 04 on all 640 projects to populate all `sentiment_score` values
2. Update `LLM_SENTIMENT` dict with the full results and re-run this notebook
3. Embed the `reports/` HTML files in the Streamlit dashboard (`app/`)